# KLTN — V2.0: ST-GCN PyTorch (Đóng góp học thuật chính)

**Mục tiêu:** Vượt ngưỡng F1 ≥ 0,78 (so V1.4 = 0,7568 ± 0,0342) bằng kiến trúc tích chập đồ thị không-thời gian.

## Kiến trúc V2.0

```
Input (B, 3, T=90, V=17)
   ↓ (channels = x, y, confidence)
Data BatchNorm
   ↓
ST-GCN Block 1: 3 → 64,   stride=1   (T=90 → 90, V=17)
   ↓ CTR-GC + Temporal Conv 9
ST-GCN Block 2: 64 → 128, stride=2   (T=90 → 45)
   ↓
ST-GCN Block 3: 128 → 256, stride=2  (T=45 → 23)
   ↓
Global Average Pool over (T, V)
   ↓
Dropout(0.5)
   ↓
Linear(256, 2)
```

**Tham chiếu:**
- Yan et al. (AAAI 2018) — ST-GCN nguyên bản
- Chen et al. (ICCV 2021) — CTR-GC refinement

**So sánh:** Multi-seed (5 seeds) với StratifiedGroupKFold giống V1.4.

## Trước khi chạy
- Folder `Human-Reco/processed_data1/` (56 feat) phải có trên Drive
- Notebook chỉ dùng 34 cột đầu (relative_kp đã hip-centered), reshape thành đồ thị 17 joint

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
ROOT_DIR = "/content/drive/MyDrive/KLTN/Human-Reco"   # ★ Sửa nếu khác
OUT_DIR = f"{ROOT_DIR}/training_outputs/v2.0_stgcn"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"OUT_DIR = {OUT_DIR}")

In [ ]:
# PyTorch đã có sẵn trên Colab
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

In [ ]:
import os, json, time, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
print("Imports OK")

## 2. Đồ thị COCO-17 và Spatial Partitioning

COCO-17 keypoints:
- 0: nose, 1-2: eyes, 3-4: ears
- 5-6: shoulders, 7-8: elbows, 9-10: wrists
- 11-12: hips, 13-14: knees, 15-16: ankles

**Spatial partitioning (Yan AAAI 2018):** 3 tập con
- Identity: chính đỉnh đó
- Centripetal: đỉnh gần trung tâm hơn (theo BFS từ nose)
- Centrifugal: đỉnh xa trung tâm hơn

In [ ]:
COCO_17_EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4),           # head
    (5, 0), (6, 0),                            # shoulder-nose
    (5, 6),                                    # shoulder-shoulder
    (5, 7), (7, 9), (6, 8), (8, 10),          # arms
    (5, 11), (6, 12), (11, 12),               # torso
    (11, 13), (13, 15), (12, 14), (14, 16),   # legs
]
V_JOINTS = 17
CENTER_JOINT = 0   # nose


def bfs_dist(adj, root):
    V = adj.shape[0]
    dist = -np.ones(V, dtype=np.int32)
    dist[root] = 0
    frontier = [root]
    while frontier:
        nf = []
        for u in frontier:
            for v in range(V):
                if adj[u, v] > 0 and dist[v] == -1:
                    dist[v] = dist[u] + 1
                    nf.append(v)
        frontier = nf
    if (dist == -1).any():
        dist[dist == -1] = dist.max() + 1
    return dist


def _norm(A):
    """D^(-1/2) A D^(-1/2)"""
    D = A.sum(0)
    Di = np.zeros_like(D, dtype=np.float32)
    Di[D > 0] = D[D > 0] ** -0.5
    return np.diag(Di) @ A @ np.diag(Di)


def build_adjacency():
    """Trả về (K=3, V, V) — 3 spatial partitions theo Yan AAAI 2018."""
    A = np.zeros((V_JOINTS, V_JOINTS), dtype=np.float32)
    for i, j in COCO_17_EDGES:
        A[i, j] = 1; A[j, i] = 1
    I = np.eye(V_JOINTS, dtype=np.float32)
    dist = bfs_dist(A, CENTER_JOINT)

    centripetal = np.zeros_like(A)
    centrifugal = np.zeros_like(A)
    for i, j in COCO_17_EDGES:
        if dist[i] == dist[j]:
            centripetal[i, j] = 1; centripetal[j, i] = 1
        elif dist[i] > dist[j]:
            centripetal[i, j] = 1
            centrifugal[j, i] = 1
        else:
            centrifugal[i, j] = 1
            centripetal[j, i] = 1

    return np.stack([_norm(I), _norm(centripetal), _norm(centrifugal)], axis=0)


A_mat = build_adjacency()
print(f"Adjacency shape: {A_mat.shape}")
print(f"Sum identity: {A_mat[0].sum():.2f}")
print(f"Sum centripetal: {A_mat[1].sum():.2f}")
print(f"Sum centrifugal: {A_mat[2].sum():.2f}")

## 3. ST-GCN block với CTR-GC refinement

**CTR-GC (Chen ICCV 2021):** mỗi channel học topology riêng cho phép graph linh hoạt hơn topology cố định của Yan 2018.

In [ ]:
class CTRGraphConv(nn.Module):
    """Channel-wise Topology Refinement Graph Convolution."""
    def __init__(self, in_ch, out_ch, A, use_ctr=True):
        super().__init__()
        K, V, _ = A.shape
        self.K, self.V = K, V
        self.register_buffer("A", torch.from_numpy(A).float())
        self.conv = nn.Conv2d(in_ch, out_ch * K, kernel_size=1)
        self.use_ctr = use_ctr
        if use_ctr:
            mid = max(out_ch // 8, 8)
            self.theta = nn.Conv2d(in_ch, mid, 1)
            self.phi = nn.Conv2d(in_ch, mid, 1)
            self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x: (N, C, T, V)
        N, _, T, V = x.shape
        y = self.conv(x)
        out_ch = y.shape[1] // self.K
        y = y.view(N, self.K, out_ch, T, V)
        out = torch.einsum("nkctv,kvw->nkctw", y, self.A)

        if self.use_ctr:
            theta = self.theta(x).mean(2)         # (N, mid, V)
            phi = self.phi(x).mean(2)
            offset = torch.tanh(torch.einsum("ncv,ncw->nvw", theta, phi))
            out_r = torch.einsum("nkctv,nvw->nkctw", y, offset)
            out = out + self.alpha * out_r
        return out.sum(1)


class TemporalConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=9, stride=1):
        super().__init__()
        pad = (kernel - 1) // 2
        self.conv = nn.Conv2d(in_ch, out_ch, (kernel, 1),
                              padding=(pad, 0), stride=(stride, 1))
        self.bn = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        return self.bn(self.conv(x))


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, t_stride=1, dropout=0.2, use_ctr=True):
        super().__init__()
        self.spatial = CTRGraphConv(in_ch, out_ch, A, use_ctr=use_ctr)
        self.bn_s = nn.BatchNorm2d(out_ch)
        self.temporal = TemporalConv(out_ch, out_ch, 9, t_stride)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout)
        if in_ch == out_ch and t_stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=(t_stride, 1)),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        res = self.residual(x)
        y = self.relu(self.bn_s(self.spatial(x)))
        y = self.temporal(y)
        y = self.dropout(y)
        return self.relu(y + res)


class STGCN(nn.Module):
    def __init__(self, in_ch=3, n_classes=2, channels=(64, 128, 256), dropout=0.5):
        super().__init__()
        A = build_adjacency()
        c1, c2, c3 = channels
        self.data_bn = nn.BatchNorm1d(in_ch * V_JOINTS)
        self.b1 = STGCNBlock(in_ch, c1, A, t_stride=1, dropout=0.2)
        self.b2 = STGCNBlock(c1, c2, A, t_stride=2, dropout=0.2)
        self.b3 = STGCNBlock(c2, c3, A, t_stride=2, dropout=0.2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(c3, n_classes)

    def forward(self, x):
        # x: (N, C, T, V)
        N, C, T, V = x.shape
        x_bn = x.permute(0, 1, 3, 2).contiguous().view(N, C * V, T)
        x_bn = self.data_bn(x_bn)
        x = x_bn.view(N, C, V, T).permute(0, 1, 3, 2).contiguous()
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = x.mean(dim=(2, 3))   # global pool (T, V)
        x = self.dropout(x)
        return self.fc(x)


# Quick test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_test = STGCN().to(device)
dummy = torch.randn(2, 3, 90, 17).to(device)
out = model_test(dummy)
print(f"Output shape: {out.shape}")
print(f"Total params: {sum(p.numel() for p in model_test.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in model_test.parameters() if p.requires_grad):,}")

## 4. Load data + chuyển sang định dạng ST-GCN

Lấy 34 cột đầu của 56 feat (relative_kp đã hip-centered), reshape → (T, V=17, C=2), pad confidence=1 → (T, V=17, C=3), permute → (C, T, V).

In [ ]:
def regenerate_skeleton(processed_root, seq_length=90, step=30):
    X, y, cids = [], [], []
    for lbl in ['normal', 'shoplifting']:
        v = 0 if lbl == 'normal' else 1
        for fp in sorted(glob.glob(os.path.join(processed_root, lbl, "*.csv"))):
            cid = f"{lbl}/{os.path.basename(fp)}"
            d = pd.read_csv(fp).values
            if d.shape[1] < 34: continue
            d34 = d[:, :34]  # lấy 34 cột đầu (relative_kp)
            for i in range(0, len(d34) - seq_length + 1, step):
                X.append(d34[i: i + seq_length])
                y.append(v)
                cids.append(cid)
    X = np.array(X, dtype='float32')      # (N, T=90, 34)
    # Reshape → (N, T, V=17, C=2)
    X = X.reshape(X.shape[0], X.shape[1], 17, 2)
    # Pad confidence channel = 1
    conf = np.ones((X.shape[0], X.shape[1], 17, 1), dtype='float32')
    X = np.concatenate([X, conf], axis=-1)  # (N, T, V, 3)
    # Permute → (N, C=3, T, V=17)
    X = X.transpose(0, 3, 1, 2)
    return X, np.array(y, dtype='int64'), np.array(cids)


X, y, clip_ids = regenerate_skeleton(f"{ROOT_DIR}/processed_data1")
print(f"X shape (N, C, T, V) = {X.shape}")
print(f"y shape = {y.shape}")
print(f"Num clips = {len(np.unique(clip_ids))}")
u, c = np.unique(y, return_counts=True)
print(f"Class dist: {dict(zip(u.tolist(), c.tolist()))}")

## 5. Dataset + augmentation

In [ ]:
class SkeletonDataset(Dataset):
    def __init__(self, X, y, indices, training=True, noise_std=0.02, crop_prob=0.3):
        self.X = X; self.y = y
        self.indices = indices
        self.training = training
        self.noise_std = noise_std
        self.crop_prob = crop_prob

    def __len__(self):
        return len(self.indices)

    def _augment(self, x):
        # x: (C, T, V) ndarray
        x = x.copy()
        if np.random.rand() < 0.8:
            # noise on x, y channels only (not confidence)
            x[:2] = x[:2] + np.random.normal(0, self.noise_std, x[:2].shape).astype(np.float32)
        if np.random.rand() < self.crop_prob:
            T = x.shape[1]
            c = np.random.randint(60, T + 1)
            if c < T:
                st = np.random.randint(0, T - c + 1)
                cr = x[:, st: st + c, :]
                i_src = np.linspace(0, c - 1, num=T)
                ifl = np.floor(i_src).astype(int); ic = np.minimum(ifl + 1, c - 1)
                w = (i_src - ifl).astype(np.float32)
                # interpolate along T
                x = (1 - w[None, :, None]) * cr[:, ifl, :] + w[None, :, None] * cr[:, ic, :]
        return x

    def __getitem__(self, i):
        idx = self.indices[i]
        x = self.X[idx]
        if self.training:
            x = self._augment(x)
        return torch.from_numpy(x.astype(np.float32)), int(self.y[idx])

## 6. Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=-1)
        ce = F.nll_loss(logp, target, reduction='none')
        pt = logp.exp().gather(1, target.unsqueeze(1)).squeeze(1)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()

## 7. Hàm train + eval cho 1 seed

In [ ]:
def split_3way_stratified(X, y, groups, seed):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    folds = list(sgkf.split(X, y, groups))
    test_fold = seed % 5
    val_fold = (seed + 1) % 5
    idx_test = folds[test_fold][1]
    idx_val  = folds[val_fold][1]
    mask = np.ones(len(X), dtype=bool)
    mask[idx_test] = False; mask[idx_val] = False
    idx_train = np.where(mask)[0]
    return idx_train, idx_val, idx_test


def train_one_seed_v20(X, y, clip_ids, seed, out_dir, epochs=120, batch_size=32, verbose=True):
    print(f"\n{'='*60}\nSEED = {seed} (V2.0 ST-GCN)\n{'='*60}")
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    idx_tr, idx_v, idx_t = split_3way_stratified(X, y, clip_ids, seed)
    # Assert
    assert not (set(clip_ids[idx_tr]) & set(clip_ids[idx_v]))
    assert not (set(clip_ids[idx_tr]) & set(clip_ids[idx_t]))
    print(f"Train={len(idx_tr)} | Val={len(idx_v)} | Test={len(idx_t)}")
    u, c = np.unique(y[idx_t], return_counts=True)
    print(f"Test class dist: {dict(zip(u.tolist(), c.tolist()))}")

    tr_ds = SkeletonDataset(X, y, idx_tr, training=True)
    v_ds  = SkeletonDataset(X, y, idx_v, training=False)
    t_ds  = SkeletonDataset(X, y, idx_t, training=False)
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    v_loader  = DataLoader(v_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    t_loader  = DataLoader(t_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = STGCN(in_ch=3, n_classes=2).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optim, T_0=10, T_mult=2)
    criterion = FocalLoss(0.25, 2.0)

    best_val_f1 = 0.0
    best_state = None
    patience = 35
    bad = 0
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "train_acc": [], "val_acc": []}

    t0 = time.time()
    for ep in range(epochs):
        # Train
        model.train()
        train_loss = 0; train_correct = 0; train_n = 0
        for xb, yb in tr_loader:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            optim.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            train_loss += loss.item() * xb.size(0)
            train_correct += (logits.argmax(-1) == yb).sum().item()
            train_n += xb.size(0)
        sched.step()
        tr_l = train_loss / train_n; tr_a = train_correct / train_n

        # Val
        model.eval()
        val_loss = 0; val_n = 0
        all_pred, all_true = [], []
        with torch.no_grad():
            for xb, yb in v_loader:
                xb = xb.to(device); yb = yb.to(device)
                logits = model(xb)
                val_loss += criterion(logits, yb).item() * xb.size(0)
                val_n += xb.size(0)
                all_pred.append(logits.argmax(-1).cpu().numpy())
                all_true.append(yb.cpu().numpy())
        v_l = val_loss / val_n
        v_a = (np.concatenate(all_pred) == np.concatenate(all_true)).mean()
        v_f = f1_score(np.concatenate(all_true), np.concatenate(all_pred), average='macro')

        history["train_loss"].append(tr_l); history["val_loss"].append(v_l)
        history["train_acc"].append(tr_a); history["val_acc"].append(v_a)
        history["val_f1"].append(v_f)

        if verbose and (ep + 1) % 10 == 0:
            print(f"  Ep {ep+1:3d}: tr_loss={tr_l:.4f} tr_acc={tr_a:.4f} | "
                  f"val_loss={v_l:.4f} val_acc={v_a:.4f} val_f1={v_f:.4f}")

        if v_f > best_val_f1:
            best_val_f1 = v_f
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"  Early stop at epoch {ep+1}")
                break

    train_time = time.time() - t0

    # Restore best
    model.load_state_dict(best_state)
    # Test eval
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for xb, yb in t_loader:
            xb = xb.to(device)
            logits = model(xb)
            all_pred.append(logits.argmax(-1).cpu().numpy())
            all_true.append(yb.numpy())
    y_pred = np.concatenate(all_pred); y_true = np.concatenate(all_true)
    f1m = f1_score(y_true, y_pred, average='macro')
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    log = {
        "seed": seed,
        "model": "ST-GCN (3 block CTR-GC, ~570K params)",
        "epochs_trained": len(history["train_loss"]),
        "train_time_sec": train_time,
        "train_size": int(len(idx_tr)), "val_size": int(len(idx_v)), "test_size": int(len(idx_t)),
        "test_f1_macro": float(f1m),
        "test_accuracy": float(acc),
        "test_precision_macro": float(prec),
        "test_recall_macro": float(rec),
        "test_confusion_matrix": cm.tolist(),
        "history": history,
    }
    torch.save(model.state_dict(), f"{out_dir}/best_seed{seed}.pt")
    with open(f"{out_dir}/log_seed{seed}.json", "w") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)
    print(f"\n[seed {seed}] F1={f1m:.4f}, Acc={acc:.4f}, epochs={len(history['train_loss'])}, time={train_time:.1f}s")
    return log

## 8. Chạy 5 seed (~60-90 phút trên T4)

In [ ]:
SEEDS = [42, 123, 7, 2024, 999]
results = []
t0 = time.time()
for s in SEEDS:
    log = train_one_seed_v20(X, y, clip_ids, s, OUT_DIR, epochs=120, batch_size=32, verbose=False)
    results.append(log)
    print(f"  → Done seed {s}, tổng {(time.time()-t0)/60:.1f} phút\n")
print(f"\nTổng: {(time.time()-t0)/60:.1f} phút")

## 9. Phân tích thống kê + so sánh V1.4 vs V2.0

In [ ]:
def stat(vals):
    return {"mean": float(np.mean(vals)), "std": float(np.std(vals, ddof=1)),
            "min": float(np.min(vals)), "max": float(np.max(vals))}

f1s   = [r["test_f1_macro"] for r in results]
accs  = [r["test_accuracy"] for r in results]
precs = [r["test_precision_macro"] for r in results]
recs  = [r["test_recall_macro"] for r in results]

summary = {
    "experiment": "V2.0 ST-GCN PyTorch + StratifiedGroupKFold + Focal Loss + Augmentation",
    "seeds": SEEDS,
    "model_summary": "ST-GCN 3-block (3→64→128→256) with CTR-GC, ~570K params",
    "f1_macro": stat(f1s), "accuracy": stat(accs),
    "precision_macro": stat(precs), "recall_macro": stat(recs),
    "individual_runs": [
        {"seed": r["seed"], "f1": r["test_f1_macro"], "acc": r["test_accuracy"],
         "epochs": r["epochs_trained"]} for r in results
    ],
}

print("="*70)
print("V2.0 ST-GCN MULTI-SEED SUMMARY")
print("="*70)
print(f"{'Metric':<20} {'Mean':>10} {'Std':>8} {'Min':>10} {'Max':>10}")
print("-"*70)
for k, v in [("F1-macro", summary["f1_macro"]), ("Accuracy", summary["accuracy"]),
             ("Precision", summary["precision_macro"]), ("Recall", summary["recall_macro"])]:
    print(f"{k:<20} {v['mean']:>10.4f} {v['std']:>8.4f} {v['min']:>10.4f} {v['max']:>10.4f}")
print("\nPer-seed:")
for r in results:
    print(f"  seed {r['seed']:>4d}: F1 = {r['test_f1_macro']:.4f}, epochs = {r['epochs_trained']}")

# So sánh với V1.4
v14 = {"mean": 0.7568, "std": 0.0342}
v20 = summary["f1_macro"]
print(f"\n=== V1.4 (LSTM-CNN) vs V2.0 (ST-GCN) ===")
print(f"V1.4: {v14['mean']:.4f} ± {v14['std']:.4f}")
print(f"V2.0: {v20['mean']:.4f} ± {v20['std']:.4f}")
print(f"Δ mean: {v20['mean'] - v14['mean']:+.4f}")
print()
if v20['mean'] >= 0.85:
    print("🎉 V2.0 ĐẠT NGƯỠNG XUẤT SẮC (≥ 0.85)! Tiếp V2.1 Temporal Transformer hoặc V2.2 Ensemble.")
elif v20['mean'] >= 0.80:
    print("✓ V2.0 vượt 0.80! Tiếp V2.1 hoặc V2.2 để vượt 0.85.")
elif v20['mean'] >= 0.78:
    print("✓ V2.0 ĐẠT NGƯỠNG TỐI THIỂU đề cương (≥ 0.78)!")
elif v20['mean'] > v14['mean']:
    print(f"~ V2.0 cải thiện so V1.4 nhưng chưa đạt 0.78. Tiếp V2.1 hoặc augment mạnh hơn.")
else:
    print(f"⚠ V2.0 chưa hơn V1.4. Cần debug (lr, scheduler, augmentation).")

with open(f"{OUT_DIR}/V2.0_multiseed_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\n✓ Saved {OUT_DIR}/V2.0_multiseed_summary.json")

In [ ]:
# Plot so sánh
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Per-seed bar comparison
v14_seeds = {42: 0.7591, 123: 0.7650, 7: 0.8050, 2024: 0.7108, 999: 0.7439}
v20_seeds = {r["seed"]: r["test_f1_macro"] for r in results}

x = np.arange(len(SEEDS))
w = 0.35
axes[0].bar(x - w/2, [v14_seeds[s] for s in SEEDS], w, label='V1.4 LSTM-CNN',
            color='lightcoral', edgecolor='black')
axes[0].bar(x + w/2, [v20_seeds[s] for s in SEEDS], w, label='V2.0 ST-GCN',
            color='steelblue', edgecolor='black')
axes[0].axhline(y=0.78, color='red', linestyle='--', alpha=0.5, label='Min 0,78')
axes[0].axhline(y=0.88, color='green', linestyle='--', alpha=0.5, label='Excellent 0,88')
axes[0].set_xticks(x); axes[0].set_xticklabels([f"seed={s}" for s in SEEDS])
axes[0].set_ylabel('F1-macro'); axes[0].set_title('V1.4 (LSTM-CNN) vs V2.0 (ST-GCN)')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.5, 1.0])

# Box plot
axes[1].boxplot([list(v14_seeds.values()), list(v20_seeds.values())],
                labels=['V1.4', 'V2.0'], showmeans=True)
axes[1].axhline(y=0.78, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(y=0.88, color='green', linestyle='--', alpha=0.5)
axes[1].set_title('Phân phối F1 — V1.4 vs V2.0')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim([0.5, 1.0])

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/v1.4_vs_v2.0.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved plot")

## 10. Bước kế tiếp

Gửi tôi `V2.0_multiseed_summary.json`.

| V2.0 Mean F1 | V2.0 Std | Quyết định |
|---:|---:|---|
| ≥ 0,85 | ≤ 0,03 | V2.0 đã xuất sắc. Có thể skip V2.1 → đi V2.2 Ensemble |
| 0,80 - 0,84 | ≤ 0,04 | Vượt 0,78 đẹp. Tiếp V2.1 (Transformer) hoặc V2.2 (Ensemble) |
| 0,78 - 0,79 | ≤ 0,05 | Đạt ngưỡng tối thiểu. Bắt buộc tiếp V2.1 hoặc V2.2 |
| 0,75 - 0,77 | — | Tốt hơn V1.4 nhưng chưa đạt. Cần V2.1 |
| < 0,75 | — | Tệ hơn dự kiến. Debug code (có thể lỗi reshape/normalize) |

Báo cáo Bảng 3.2 sẽ có thêm dòng:
- "V2.0 — ST-GCN mean (5 runs)"
- "V2.0 — ST-GCN std (5 runs)"

Bảng 3.3 (chi tiết 5 seed) sẽ mở rộng thành cột V1.4 vs V2.0.